# Step 2 V2 — Federated Training: All Baselines + Sweeps

**NeurIPS additions over V1:**

| Experiment | Description |
|---|---|
| `fedavg_noproj` | FedAvg, no safety constraint (V1 baseline) |
| `fedavg_proj` | FedAvg + safety gradient projection (V1 proposed) |
| `fedprox` | FedProx (μ=0.01), no projection |
| `fedprox_proj` | FedProx + projection (**new strongest baseline**) |
| Adversarial fraction sweep | 10%, 30%, 50%, 70%, 90% biased clients |
| Non-IID sweep | IID, Dirichlet α=0.5, α=0.1 |
| Multi-seed | 5 seeds per configuration |
| Gradient alignment tracking | Cosine sim to safety subspace per round |

**Produces:**
- `models/step2_{method}_seed{s}.pt` — per-method per-seed LoRA weights
- `results/step2_all_results.pt` — full results dict
- `results/step2_adv_sweep.pt` — adversarial fraction sweep
- `results/step2_noniid_sweep.pt` — non-IID sweep
- `results/step2_alignment_tracking.pt` — gradient alignment per round

In [ ]:
import sys
print('=' * 60)
print('ENVIRONMENT')
print('=' * 60)
print(f'Python: {sys.version}')
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name} ({p.total_memory/1024**3:.1f} GB)')
print('=' * 60)

In [ ]:
%pip install -q transformers datasets peft evaluate matplotlib

In [ ]:
import sys
sys.path.insert(0, '.')
from utils import *
import matplotlib.pyplot as plt
from peft import PeftModel

# ── Step 2 experiment grid ─────────────────────────────────────────────────────
METHODS = [
    {'name': 'fedavg_noproj',   'use_proj': False, 'mu': 0.0},
    {'name': 'fedavg_proj',     'use_proj': True,  'mu': 0.0},
    {'name': 'fedprox',         'use_proj': False, 'mu': 0.01},
    {'name': 'fedprox_proj',    'use_proj': True,  'mu': 0.01},
]

# Adversarial fraction: fraction of clients using REJECTED (biased) data
# 1.0 = all biased (V1 default), 0.5 = 5 biased + 5 clean
ADV_FRACTIONS     = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]   # main sweep
NONIID_ALPHAS     = ['iid', 0.5, 0.1]                  # IID or Dirichlet alpha

MAIN_ADV_FRACTION = 1.0    # default for method comparison
MAIN_N_ROUNDS     = 10
MAIN_N_CLIENTS    = 10
SEEDS_MAIN        = SEEDS  # all 5 seeds for main results
SEEDS_SWEEP       = SEEDS[:3]  # 3 seeds for sweeps

GPU_ID = 0
_DEVICE = torch.device(f'cuda:{GPU_ID}' if torch.cuda.is_available() else 'cpu')

print('Methods:', [m['name'] for m in METHODS])
print(f'Adversarial fractions: {ADV_FRACTIONS}')
print(f'Non-IID alphas: {NONIID_ALPHAS}')
print(f'Main seeds: {SEEDS_MAIN}')
print(f'Device: {_DEVICE}')

In [ ]:
def make_client_data(rejected_texts, chosen_texts, tokenizer,
                     adv_fraction, n_clients, seed, alpha='iid'):
    """
    Build per-client datasets.
    adv_fraction: fraction of clients that receive rejected (biased) texts.
    Remaining clients receive chosen (clean) texts.
    alpha: 'iid' for uniform split, float for Dirichlet non-IID.
    """
    n_adv = max(1, int(n_clients * adv_fraction))
    n_clean = n_clients - n_adv

    if alpha == 'iid':
        adv_ds = prepare_client_datasets_iid(rejected_texts, tokenizer, n_adv, seed)
        clean_ds = prepare_client_datasets_iid(chosen_texts, tokenizer, n_clean, seed + 1) if n_clean > 0 else []
    else:
        adv_ds = prepare_client_datasets_noniid(rejected_texts, tokenizer, n_adv, float(alpha), seed)
        clean_ds = prepare_client_datasets_noniid(chosen_texts, tokenizer, n_clean, float(alpha), seed + 1) if n_clean > 0 else []

    return adv_ds + clean_ds


def run_fl_experiment(
    model, tokenizer, client_datasets, safety_bases,
    n_rounds, use_proj, mu, device,
    track_alignment=False,
):
    """
    Core FL simulation. Returns (final_model, round_losses, alignment_per_round).
    alignment_per_round: gradient alignment to safety subspace per round (only if track_alignment).
    """
    global_state = get_lora_state(model)
    round_losses, alignment_per_round = [], []

    for rnd in range(n_rounds):
        client_states, client_losses, client_alignments = [], [], []

        for cid, ds in enumerate(client_datasets):
            set_lora_state(model, global_state)

            if use_proj:
                handles = register_safety_hooks(model, safety_bases)

            # One epoch of local training
            loader = torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE,
                                                  shuffle=True, num_workers=0)
            optimizer = torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad], lr=LR_FINETUNE)
            model.train()
            total_loss, n = 0.0, 0

            for batch in loader:
                ids  = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                lbls = batch['labels'].to(device)
                out  = model(input_ids=ids, attention_mask=mask, labels=lbls)
                loss = out.loss

                if mu > 0.0:
                    prox = sum(
                        (p - global_state[n_].to(device, dtype=p.dtype)).pow(2).sum()
                        for n_, p in model.named_parameters() if n_ in global_state
                    )
                    loss = loss + (mu / 2.0) * prox

                optimizer.zero_grad()
                loss.backward()

                # Measure alignment before hooks remove the gradient components
                if track_alignment and cid == 0:
                    align = compute_gradient_alignment(model, safety_bases)
                    client_alignments.append(align)

                optimizer.step()
                total_loss += out.loss.item()
                n += 1

            if use_proj:
                remove_hooks(handles)

            client_states.append(get_lora_state(model))
            client_losses.append(total_loss / max(n, 1))

        global_state = average_lora_states(client_states)
        avg_loss = np.mean(client_losses)
        round_losses.append(float(avg_loss))

        if client_alignments:
            alignment_per_round.append(float(np.mean(client_alignments)))

        torch.cuda.empty_cache()
        print(f'  Round {rnd+1}/{n_rounds}: avg_loss={avg_loss:.4f}'
              + (f'  alignment={alignment_per_round[-1]:.4f}' if alignment_per_round else ''))

    set_lora_state(model, global_state)
    return model, round_losses, alignment_per_round

print('FL core functions defined.')

In [ ]:
# ── Main method comparison (all seeds × all methods) ──────────────────────────
# Uses: adv_fraction=1.0 (all clients biased), IID, N_ROUNDS=10, N_CLIENTS=10

main_results_path = RESULTS_DIR / 'step2_main_results.pt'

if main_results_path.exists():
    main_results = torch.load(str(main_results_path))
    print(f'Loaded existing main results ({len(main_results)} entries).')
else:
    main_results = []

done_keys = {(r['method'], r['seed']) for r in main_results}

for seed in SEEDS_MAIN:
    print(f'\n{"#"*60}')
    print(f'SEED {seed}')
    print(f'{"#"*60}')

    # Load data for this seed
    rejected_path = DATA_DIR / f'rejected_texts_seed{seed}.pt'
    chosen_path   = DATA_DIR / f'chosen_texts_seed{seed}.pt'
    subspace_path = MODELS_DIR / f'safety_subspace_seed{seed}.pt'
    lora_path     = MODELS_DIR / f'safety_lora_seed{seed}'

    if not rejected_path.exists():
        print(f'  Data not found for seed {seed} — run Step 1 first.')
        continue

    rejected_texts = torch.load(str(rejected_path))
    chosen_texts, _ = load_hh_rlhf(n_chosen=N_SAFETY_SAMPLES, n_rejected=0, seed=seed)
    torch.save(chosen_texts, str(chosen_path))

    ss = SafetySubspace.load(subspace_path)
    safety_bases = ss.bases

    for method_cfg in METHODS:
        mname = method_cfg['name']
        use_proj = method_cfg['use_proj']
        mu = method_cfg['mu']

        if (mname, seed) in done_keys:
            print(f'  {mname} seed={seed}: already done, skipping.')
            continue

        print(f'\n  Method: {mname}  seed={seed}')
        save_path = MODELS_DIR / f'step2_{mname}_seed{seed}.pt'

        # Load model fresh from safety LoRA starting point
        base, tokenizer = load_base_model_and_tokenizer(gpu_id=GPU_ID)
        if lora_path.exists():
            model = PeftModel.from_pretrained(base, str(lora_path), is_trainable=True)
        else:
            model = apply_lora(base)

        client_datasets = make_client_data(
            rejected_texts, chosen_texts, tokenizer,
            adv_fraction=MAIN_ADV_FRACTION,
            n_clients=MAIN_N_CLIENTS, seed=seed,
        )

        model, round_losses, alignment = run_fl_experiment(
            model, tokenizer, client_datasets, safety_bases,
            n_rounds=MAIN_N_ROUNDS, use_proj=use_proj, mu=mu,
            device=_DEVICE, track_alignment=True,
        )

        final_state = get_lora_state(model)
        torch.save(final_state, str(save_path))

        entry = {
            'method': mname, 'seed': seed, 'use_proj': use_proj, 'mu': mu,
            'round_losses': round_losses, 'alignment_per_round': alignment,
            'final_loss': round_losses[-1] if round_losses else None,
            'save_path': str(save_path),
        }
        main_results.append(entry)
        done_keys.add((mname, seed))
        torch.save(main_results, str(main_results_path))
        print(f'  Saved: {save_path.name}')

        del base, model
        torch.cuda.empty_cache()

print(f'\nMain comparison done: {len(main_results)} runs.')

In [ ]:
# ── Adversarial fraction sweep ─────────────────────────────────────────────────
# Fix method=fedavg_proj vs fedavg_noproj, vary adv_fraction, 3 seeds

adv_sweep_path = RESULTS_DIR / 'step2_adv_sweep.pt'

if adv_sweep_path.exists():
    adv_sweep_results = torch.load(str(adv_sweep_path))
    print(f'Loaded adversarial sweep ({len(adv_sweep_results)} entries).')
else:
    adv_sweep_results = []

done_adv = {(r['method'], r['adv_fraction'], r['seed']) for r in adv_sweep_results}
SWEEP_METHODS = [m for m in METHODS if m['name'] in ('fedavg_noproj', 'fedavg_proj')]

for seed in SEEDS_SWEEP:
    rejected_path = DATA_DIR / f'rejected_texts_seed{seed}.pt'
    chosen_path   = DATA_DIR / f'chosen_texts_seed{seed}.pt'
    if not rejected_path.exists():
        continue
    rejected_texts = torch.load(str(rejected_path))
    chosen_texts, _ = load_hh_rlhf(n_chosen=N_SAFETY_SAMPLES, n_rejected=0, seed=seed)

    ss = SafetySubspace.load(MODELS_DIR / f'safety_subspace_seed{seed}.pt')
    safety_bases = ss.bases
    lora_path = MODELS_DIR / f'safety_lora_seed{seed}'

    for frac in ADV_FRACTIONS:
        for method_cfg in SWEEP_METHODS:
            mname = method_cfg['name']
            if (mname, frac, seed) in done_adv:
                print(f'  {mname} frac={frac} seed={seed}: skip')
                continue

            print(f'\n  {mname} | adv_frac={frac} | seed={seed}')
            base, tokenizer = load_base_model_and_tokenizer(gpu_id=GPU_ID)
            if lora_path.exists():
                model = PeftModel.from_pretrained(base, str(lora_path), is_trainable=True)
            else:
                model = apply_lora(base)

            client_datasets = make_client_data(
                rejected_texts, chosen_texts, tokenizer,
                adv_fraction=frac, n_clients=MAIN_N_CLIENTS, seed=seed,
            )
            model, round_losses, alignment = run_fl_experiment(
                model, tokenizer, client_datasets, safety_bases,
                n_rounds=MAIN_N_ROUNDS, use_proj=method_cfg['use_proj'],
                mu=method_cfg['mu'], device=_DEVICE, track_alignment=False,
            )

            adv_sweep_results.append({
                'method': mname, 'adv_fraction': frac, 'seed': seed,
                'final_loss': round_losses[-1],
                'round_losses': round_losses,
            })
            done_adv.add((mname, frac, seed))
            torch.save(adv_sweep_results, str(adv_sweep_path))

            del base, model
            torch.cuda.empty_cache()

print('Adversarial fraction sweep complete.')

In [ ]:
# ── Non-IID sweep ──────────────────────────────────────────────────────────────
noniid_path = RESULTS_DIR / 'step2_noniid_sweep.pt'

if noniid_path.exists():
    noniid_results = torch.load(str(noniid_path))
    print(f'Loaded non-IID sweep ({len(noniid_results)} entries).')
else:
    noniid_results = []

done_noniid = {(r['method'], str(r['alpha']), r['seed']) for r in noniid_results}

for seed in SEEDS_SWEEP:
    rejected_path = DATA_DIR / f'rejected_texts_seed{seed}.pt'
    chosen_path   = DATA_DIR / f'chosen_texts_seed{seed}.pt'
    if not rejected_path.exists():
        continue
    rejected_texts = torch.load(str(rejected_path))
    chosen_texts, _ = load_hh_rlhf(n_chosen=N_SAFETY_SAMPLES, n_rejected=0, seed=seed)
    ss = SafetySubspace.load(MODELS_DIR / f'safety_subspace_seed{seed}.pt')
    safety_bases = ss.bases
    lora_path = MODELS_DIR / f'safety_lora_seed{seed}'

    for alpha in NONIID_ALPHAS:
        for method_cfg in SWEEP_METHODS:
            mname = method_cfg['name']
            if (mname, str(alpha), seed) in done_noniid:
                print(f'  {mname} alpha={alpha} seed={seed}: skip')
                continue

            print(f'\n  {mname} | alpha={alpha} | seed={seed}')
            base, tokenizer = load_base_model_and_tokenizer(gpu_id=GPU_ID)
            if lora_path.exists():
                model = PeftModel.from_pretrained(base, str(lora_path), is_trainable=True)
            else:
                model = apply_lora(base)

            client_datasets = make_client_data(
                rejected_texts, chosen_texts, tokenizer,
                adv_fraction=MAIN_ADV_FRACTION, n_clients=MAIN_N_CLIENTS,
                seed=seed, alpha=alpha,
            )
            model, round_losses, alignment = run_fl_experiment(
                model, tokenizer, client_datasets, safety_bases,
                n_rounds=MAIN_N_ROUNDS, use_proj=method_cfg['use_proj'],
                mu=method_cfg['mu'], device=_DEVICE,
            )

            noniid_results.append({
                'method': mname, 'alpha': alpha, 'seed': seed,
                'final_loss': round_losses[-1],
                'round_losses': round_losses,
            })
            done_noniid.add((mname, str(alpha), seed))
            torch.save(noniid_results, str(noniid_path))

            del base, model
            torch.cuda.empty_cache()

print('Non-IID sweep complete.')

In [ ]:
# ── Plot: Round loss curves per method (seed=42) ──────────────────────────────
import matplotlib.pyplot as plt

main_results = torch.load(str(RESULTS_DIR / 'step2_main_results.pt'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'fedavg_noproj': '#e74c3c', 'fedavg_proj': '#2980b9',
          'fedprox': '#f39c12', 'fedprox_proj': '#27ae60'}

# Left: loss curves for seed=42
for r in main_results:
    if r['seed'] != 42:
        continue
    ax = axes[0]
    ax.plot(range(1, len(r['round_losses']) + 1), r['round_losses'],
            label=r['method'], color=colors.get(r['method'], 'gray'),
            linewidth=2, marker='o', markersize=4)
axes[0].set_xlabel('FL Round', fontsize=12)
axes[0].set_ylabel('Avg Training Loss', fontsize=12)
axes[0].set_title('Round Loss by Method (seed=42)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.4)

# Right: gradient alignment per round (seed=42, fedavg_noproj vs fedavg_proj)
for r in main_results:
    if r['seed'] != 42 or not r['alignment_per_round']:
        continue
    axes[1].plot(range(1, len(r['alignment_per_round']) + 1), r['alignment_per_round'],
                 label=r['method'], color=colors.get(r['method'], 'gray'),
                 linewidth=2, marker='s', markersize=4)
axes[1].set_xlabel('FL Round', fontsize=12)
axes[1].set_ylabel('Gradient Alignment to Safety Subspace', fontsize=12)
axes[1].set_title('Safety Subspace Alignment During FL (seed=42)', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'step2_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: step2_training_curves.png')

In [ ]:
# ── Plot: Adversarial fraction sweep ──────────────────────────────────────────
adv_results = torch.load(str(RESULTS_DIR / 'step2_adv_sweep.pt'))

fig, ax = plt.subplots(figsize=(9, 5))
for mname, color in [('fedavg_noproj', '#e74c3c'), ('fedavg_proj', '#2980b9')]:
    fracs, means, stds = [], [], []
    for frac in ADV_FRACTIONS:
        vals = [r['final_loss'] for r in adv_results
                if r['method'] == mname and r['adv_fraction'] == frac]
        if vals:
            fracs.append(frac)
            means.append(np.mean(vals))
            stds.append(np.std(vals))
    fracs, means, stds = np.array(fracs), np.array(means), np.array(stds)
    ax.plot(fracs, means, 'o-', color=color, label=mname, linewidth=2, markersize=7)
    ax.fill_between(fracs, means - stds, means + stds, alpha=0.15, color=color)

ax.set_xlabel('Adversarial Client Fraction', fontsize=12)
ax.set_ylabel('Final Round Avg Loss', fontsize=12)
ax.set_title('Robustness to Adversarial Clients (3 seeds ± std)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'step2_adv_fraction_sweep.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: step2_adv_fraction_sweep.png')

In [ ]:
main_results = torch.load(str(RESULTS_DIR / 'step2_main_results.pt'))
print('\n' + '='*65)
print(f'{"Method":<20} {"Seed":>6} {"Final Loss":>12} {"Alignment":>12}')
print('-'*65)
for r in sorted(main_results, key=lambda x: (x['method'], x['seed'])):
    align = r['alignment_per_round'][-1] if r['alignment_per_round'] else float('nan')
    print(f"{r['method']:<20} {r['seed']:>6} {r['final_loss']:>12.4f} {align:>12.4f}")
print('='*65)